# `hierarchy.py` playground

Goal: build `build_summing_matrix()`'s logic yourself, one level at a time, using the same **Region × Product** retailer example from the conversation (CA/NY × Trousers/T-shirts).

Run each cell in order. Cells marked **YOUR TURN** have no working code yet — write it yourself before moving on. A **Solutions** section at the very bottom lets you check your work — don't peek before you try.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", None)

## Step 1 — the `series_meta` table

This is the input `build_summing_matrix()` starts from. 9 rows: 1 Total, 2 Region, 2 Product, 4 bottom (Region×Product).

In [3]:
series_meta = pd.DataFrame(
    [
        {"series_id": "s0", "level": "Total", "is_bottom": False, "region": None, "product": None},
        {"series_id": "s1", "level": "Region", "is_bottom": False, "region": "CA", "product": None},
        {"series_id": "s2", "level": "Region", "is_bottom": False, "region": "NY", "product": None},
        {"series_id": "s3", "level": "Product", "is_bottom": False, "region": None, "product": "Trousers"},
        {"series_id": "s4", "level": "Product", "is_bottom": False, "region": None, "product": "T-shirts"},
        {"series_id": "s5", "level": "Region/Product", "is_bottom": True, "region": "CA", "product": "Trousers"},
        {"series_id": "s6", "level": "Region/Product", "is_bottom": True, "region": "CA", "product": "T-shirts"},
        {"series_id": "s7", "level": "Region/Product", "is_bottom": True, "region": "NY", "product": "Trousers"},
        {"series_id": "s8", "level": "Region/Product", "is_bottom": True, "region": "NY", "product": "T-shirts"},
    ]
).set_index("series_id")

series_meta

,level,is_bottom,region,product
series_id,,,,
s0,Total,False,None,None
s1,Region,False,CA,None
s2,Region,False,NY,None
s3,Product,False,None,Trousers
s4,Product,False,None,T-shirts
s5,Region/Product,True,CA,Trousers
s6,Region/Product,True,CA,T-shirts
s7,Region/Product,True,NY,Trousers
s8,Region/Product,True,NY,T-shirts


## Step 2 — `dim_cols`: which columns describe groups?

Everything except `level` and `is_bottom` is a dimension column.

In [4]:
dim_cols = [c for c in series_meta.columns if c not in ("level", "is_bottom")]
dim_cols

['region', 'product']

## Step 3 — the bottom rows

These are the only *real, measured* series. Everything else is a sum of these.

In [5]:
bottom = series_meta[series_meta["is_bottom"]]
bottom

,level,is_bottom,region,product
series_id,,,,
s5,Region/Product,True,CA,Trousers
s6,Region/Product,True,CA,T-shirts
s7,Region/Product,True,NY,Trousers
s8,Region/Product,True,NY,T-shirts


## Trying to generalize it

In [6]:
for level, level_rows in series_meta.groupby("level", sort=False):
    defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]

    if not defining_cols:
        continue

    matches = bottom.reset_index().merge(
        level_rows.reset_index()[["series_id"] + defining_cols],
        on=defining_cols,
        suffixes=("_bottom", "_agg"),
    )

    print(matches[["series_id_bottom", "series_id_agg"]])

  series_id_bottom series_id_agg
0               s5            s1
1               s6            s1
2               s7            s2
3               s8            s2
  series_id_bottom series_id_agg
0               s5            s3
1               s7            s3
2               s6            s4
3               s8            s4
  series_id_bottom series_id_agg
0               s5            s5
1               s6            s6
2               s7            s7
3               s8            s8


## Step 4 — worked example: the "Region" level

For one level, the real function does two things:
1. `defining_cols` — which dimension columns actually have a value at this level (ignore the rest)
2. merge `bottom` against this level's rows **on just those columns** — that merge *is* "which bottom series belong to which aggregate"

Run this and read the output before moving on.

In [7]:
level_rows = series_meta[series_meta["level"] == "Region"]

defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]
print("defining_cols:", defining_cols)

matches = bottom.reset_index().merge(
    level_rows.reset_index()[["series_id"] + defining_cols],
    on=defining_cols,
    suffixes=("_bottom", "_agg"),
)
matches[["series_id_bottom", "series_id_agg"]]
matches

defining_cols: ['region']


,series_id_bottom,level,is_bottom,region,product,series_id_agg
0,s5,Region/Product,True,CA,Trousers,s1
1,s6,Region/Product,True,CA,T-shirts,s1
2,s7,Region/Product,True,NY,Trousers,s2
3,s8,Region/Product,True,NY,T-shirts,s2


Check: `s1` (CA) should match `s5` and `s6`. `s2` (NY) should match `s7` and `s8`. Does the table above say that?

Each `(bottom_id, agg_id)` pair above is exactly one `1` in the summing matrix `S`.

## Step 5 — YOUR TURN: the "Product" level

Do the exact same thing as Step 4, but for `level == "Product"`. Predict the answer first (which bottom series should `s3` "Trousers" match?), then write the code and check yourself.

In [8]:
level_rows = series_meta[series_meta["level"] == "Product"]
level_rows

,level,is_bottom,region,product
series_id,,,,
s3,Product,False,None,Trousers
s4,Product,False,None,T-shirts


In [9]:

level_rows = series_meta[series_meta["level"] == "Product"]
defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]
defining_cols

['product']

In [10]:
bottom

,level,is_bottom,region,product
series_id,,,,
s5,Region/Product,True,CA,Trousers
s6,Region/Product,True,CA,T-shirts
s7,Region/Product,True,NY,Trousers
s8,Region/Product,True,NY,T-shirts


In [11]:
level_rows = series_meta[series_meta["level"] == "Product"]
defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]

matches = bottom.reset_index().merge(
    level_rows.reset_index()[["series_id"] + defining_cols],
    on=defining_cols,
    suffixes=("_bottom", "_agg"),
)
matches[["series_id_bottom", "series_id_agg"]]


,series_id_bottom,series_id_agg
0,s5,s3
1,s7,s3
2,s6,s4
3,s8,s4


## Step 6 — YOUR TURN: the "Total" level (the special case)

Try the same approach for `level == "Total"` first and see what `defining_cols` comes out as. That's the case the real code special-cases with an `if not defining_cols:` branch — "every bottom series is a member." Write the code that produces `(bottom_id, "s0")` for all 4 bottom series, however you like (the merge from before won't work when `defining_cols` is empty — think about why, then do something else).

In [12]:
# YOUR TURN
level_rows = series_meta[series_meta["level"] == "Region"]

defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]

bottom = series_meta[series_meta["is_bottom"]]
col_ids = bottom.index.tolist()
print(col_ids)
col_pos = {sid: i for i, sid in enumerate(col_ids)}
print(col_pos)
row_ids = series_meta.index.tolist()
print(row_ids)
row_pos = {sid: i for i, sid in enumerate(row_ids)}
print(row_pos)

rows, cols = [], []

matches = bottom.reset_index().merge(
    level_rows.reset_index()[["series_id"] + defining_cols],
    on=defining_cols,
    suffixes=("_bottom", "_agg"),
)
print(matches)
for bottom_id, agg_id in zip(matches["series_id_bottom"], matches["series_id_agg"]):
    rows.append(row_pos[agg_id])
    cols.append(col_pos[bottom_id])
print(rows) 
print(cols)

['s5', 's6', 's7', 's8']
{'s5': 0, 's6': 1, 's7': 2, 's8': 3}
['s0', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8']
{'s0': 0, 's1': 1, 's2': 2, 's3': 3, 's4': 4, 's5': 5, 's6': 6, 's7': 7, 's8': 8}
  series_id_bottom           level  is_bottom region   product series_id_agg
0               s5  Region/Product       True     CA  Trousers            s1
1               s6  Region/Product       True     CA  T-shirts            s1
2               s7  Region/Product       True     NY  Trousers            s2
3               s8  Region/Product       True     NY  T-shirts            s2
[1, 1, 2, 2]
[0, 1, 2, 3]


## Step 7 — check yourself against the real function

Run the actual library code on the same `series_meta` and see the whole matrix at once.

In [13]:
import sys
sys.path.insert(0, "..")

from hts_bench.data.hierarchy import build_summing_matrix

S = build_summing_matrix(series_meta)

S_dense = pd.DataFrame(S.matrix.toarray(), index=S.row_ids, columns=S.col_ids)
S_dense

,s5,s6,s7,s8
s0,1,1,1,1
s1,1,1,0,0
s2,0,0,1,1
s3,1,0,1,0
s4,0,1,0,1
s5,1,0,0,0
s6,0,1,0,0
s7,0,0,1,0
s8,0,0,0,1


Compare this to the table from the conversation. Every row you should recognize: `s0` all 1s, `s1`/`s2` matching your Step 4 answer, `s3`/`s4` matching your Step 5 answer, `s5`-`s8` each a single 1 on their own column.

## Step 8 (bonus) — `aggregate_from_bottom`: use S on real numbers

Make up sales numbers for the 4 bottom series across 2 days, and watch every aggregate get computed automatically.

In [14]:
from hts_bench.data.loader import aggregate_from_bottom

bottom_data = pd.DataFrame(
    {"s5": [10.0, 12.0], "s6": [4.0, 5.0], "s7": [7.0, 6.0], "s8": [3.0, 3.0]},
    index=pd.to_datetime(["2024-01-01", "2024-01-02"]),
)
bottom_data.index.name = "date"

full = aggregate_from_bottom(S, bottom_data)
full

,s0,s1,s2,s3,s4,s5,s6,s7,s8
date,,,,,,,,,
2024-01-01,24.0,14.0,10.0,17.0,7.0,10.0,4.0,7.0,3.0
2024-01-02,26.0,17.0,9.0,18.0,8.0,12.0,5.0,6.0,3.0


Sanity checks you can do by eye: does `s0` (Total) equal `s5+s6+s7+s8` on day 1? Does `s3` (Trousers) equal `s5+s7`? Try changing a number in `bottom_data` above and re-running — watch every aggregate update.

---
## Solutions (don't peek until you've tried)

### Step 5 solution

In [15]:
level_rows = series_meta[series_meta["level"] == "Product"]
defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]
matches = bottom.reset_index().merge(
    level_rows.reset_index()[["series_id"] + defining_cols],
    on=defining_cols,
    suffixes=("_bottom", "_agg"),
)
matches[["series_id_bottom", "series_id_agg"]]

,series_id_bottom,series_id_agg
0,s5,s3
1,s7,s3
2,s6,s4
3,s8,s4


### Step 6 solution

`defining_cols` comes out `[]` (empty) because `Total`'s row has `None` in every dimension column. The merge approach breaks here (merging "on no columns" isn't meaningful), which is exactly why the real code special-cases it: when there's nothing to match on, *every* bottom series counts.

In [16]:
level_rows = series_meta[series_meta["level"] == "Total"]
defining_cols = [c for c in dim_cols if level_rows[c].notna().any()]
print("defining_cols:", defining_cols)

pairs = [(bottom_id, agg_id) for agg_id in level_rows.index for bottom_id in bottom.index]
pairs

defining_cols: []


[('s5', 's0'), ('s6', 's0'), ('s7', 's0'), ('s8', 's0')]